# AI4Bharat Indic Parler-TTS 테스트

산스크리트어 네이티브 지원 — 빠알리어 발음 품질 확인

In [ ]:
!pip install -q git+https://github.com/huggingface/parler-tts.git transformers torch soundfile
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print('설치 완료!')

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf
import torch
import IPython.display as ipd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ParlerTTSForConditionalGeneration.from_pretrained('ai4bharat/indic-parler-tts-pretrained').to(device)
tok = AutoTokenizer.from_pretrained('ai4bharat/indic-parler-tts-pretrained')
print('모델 로드 완료!')

In [ ]:
# 데바나가리 변환 함수
CONSONANTS = {
    'kh': '\u0916', 'k': '\u0915', 'gh': '\u0918', 'g': '\u0917', '\u1e45': '\u0919',
    'ch': '\u091b', 'c': '\u091a', 'jh': '\u091d', 'j': '\u091c', '\u00f1': '\u091e',
    '\u1e6dh': '\u0920', '\u1e6d': '\u091f', '\u1e0dh': '\u0922', '\u1e0d': '\u0921', '\u1e47': '\u0923',
    'th': '\u0925', 't': '\u0924', 'dh': '\u0927', 'd': '\u0926', 'n': '\u0928',
    'ph': '\u092b', 'p': '\u092a', 'bh': '\u092d', 'b': '\u092c', 'm': '\u092e',
    'y': '\u092f', 'r': '\u0930', 'l': '\u0932', '\u1e37': '\u0933',
    'v': '\u0935', 's': '\u0938', 'h': '\u0939',
}
VOWELS_IND = { '\u0101': '\u0906', 'a': '\u0905', '\u012b': '\u0908', 'i': '\u0907', '\u016b': '\u090a', 'u': '\u0909', 'e': '\u090f', 'o': '\u0913' }
VOWELS_DEP = { '\u0101': '\u093e', 'a': '', '\u012b': '\u0940', 'i': '\u093f', '\u016b': '\u0942', 'u': '\u0941', 'e': '\u0947', 'o': '\u094b' }
VIRAMA = '\u094d'

def pali_to_devanagari(roman):
    result = ''; i = 0; s = roman.lower()
    while i < len(s):
        ch = s[i]
        if ch in ' ,.;:!?-\n\r\t': result += ch; i += 1; continue
        if ch == '\u1e43': result += '\u0902'; i += 1; continue
        three = s[i:i+3]; two = s[i:i+2]
        consonant = None; consumed = 0
        if three in CONSONANTS: consonant = CONSONANTS[three]; consumed = 3
        elif two in CONSONANTS: consonant = CONSONANTS[two]; consumed = 2
        elif ch in CONSONANTS: consonant = CONSONANTS[ch]; consumed = 1
        if consonant:
            i += consumed
            if i < len(s) and s[i] in VOWELS_IND: result += consonant + VOWELS_DEP[s[i]]; i += 1
            else: result += consonant + VIRAMA
            continue
        if ch in VOWELS_IND: result += VOWELS_IND[ch]; i += 1; continue
        result += ch; i += 1
    return result

print('변환 함수 준비')

In [ ]:
# === 테스트 단어 5개 생성 + 재생 ===
test_words = ['aṅguli', 'buddha', 'dhammaṃ', 'bhikkhu', 'nibbāna']
description = 'A calm male voice speaks clearly and slowly'

for word in test_words:
    deva = pali_to_devanagari(word)
    print(f'\n{word} → {deva}')
    
    desc_input = tok(description, return_tensors='pt').to(device)
    text_input = tok(deva, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output = model.generate(input_ids=desc_input.input_ids, prompt_input_ids=text_input.input_ids)
    
    audio = output.squeeze().cpu().numpy()
    sr = model.config.sampling_rate
    print(f'  길이: {len(audio)/sr:.2f}초')
    ipd.display(ipd.Audio(audio, rate=sr))